# Exploratory Data Analysis for KDD Cup 1999

This notebook loads the corrected 10 percent KDD99 dataset, reconstructs the schema from the original KDD metadata files, and documents the distribution shifts that matter for autoencoder-based anomaly detection.


## 1. Setup

The notebook is designed to run from the `notebooks/` directory. It uses `kddcup.names` as the source of truth for feature order and `training_attack_types` to group individual attacks into KDD threat families.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Keep all raw-data dependencies together so the notebook is easy to retarget.
DATA_PATH = Path("../data/raw/kddcup.data_10_percent_corrected")
NAMES_PATH = Path("../data/raw/kddcup.names")
ATTACK_TYPES_PATH = Path("../data/raw/training_attack_types")

# Fall back to the original 10 percent export if the corrected file is unavailable.
if not DATA_PATH.exists():
    fallback_path = Path("../data/raw/kddcup.data_10_percent.gz")
    if fallback_path.exists():
        DATA_PATH = fallback_path

# Stop early if a required source file is missing.
for required_path in [DATA_PATH, NAMES_PATH, ATTACK_TYPES_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file not found: {required_path}")

print(f"Dataset path: {DATA_PATH}")
print(f"Column metadata path: {NAMES_PATH}")
print(f"Attack taxonomy path: {ATTACK_TYPES_PATH}")


## 2. Load Metadata and Build the Dataset Schema

The raw KDD Cup 1999 file does not include headers, so this notebook parses `kddcup.names` to recover both the feature order and each feature type before loading the data.


In [ ]:
def load_kdd_data_dictionary(names_path: Path) -> pd.DataFrame:
    """Parse the canonical KDD99 feature definitions from kddcup.names."""
    raw_lines = names_path.read_text().splitlines()
    feature_rows = []

    # The first line lists the label names; the feature definitions start after that.
    for line in raw_lines[1:]:
        cleaned_line = line.strip()
        if not cleaned_line or ":" not in cleaned_line:
            continue
        feature_name, feature_type = cleaned_line.split(":", maxsplit=1)
        feature_rows.append({
            "feature_name": feature_name.strip(),
            "declared_type": feature_type.strip().rstrip("."),
        })

    return pd.DataFrame(feature_rows)


feature_dictionary = load_kdd_data_dictionary(NAMES_PATH)
column_names = feature_dictionary["feature_name"].tolist() + ["label"]
categorical_features = feature_dictionary.loc[
    feature_dictionary["declared_type"].eq("symbolic"), "feature_name"
].tolist()

# This assertion guards against silent schema drift when the raw file changes.
expected_column_count = 42
if len(column_names) != expected_column_count:
    raise ValueError(
        f"Expected {expected_column_count} columns from the metadata files, found {len(column_names)}"
    )

df = pd.read_csv(DATA_PATH, header=None, names=column_names)

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
print(f"Categorical features from the data dictionary: {categorical_features}")
feature_dictionary.head()


## 3. Basic Inspection

Check the shape, data types, and summary information before any preprocessing so that schema issues show up early.


In [ ]:
# A quick column-level inspection helps confirm that the metadata-driven loader worked.
print(f"Shape: {df.shape}")
print()
print("Data types:")
print(df.dtypes)
print()
df.info()


## 4. Attack Sub-Category Grouping

The dataset contains specific attack labels, but modeling decisions are easier to reason about when those labels are grouped into the four standard KDD threat families: DoS, Probe, R2L, and U2R.


In [ ]:
def load_attack_family_map(attack_types_path: Path) -> dict[str, str]:
    """Map each attack label to its broader KDD attack family."""
    family_name_map = {"dos": "DoS", "probe": "Probe", "r2l": "R2L", "u2r": "U2R"}
    attack_family_map = {"normal.": "normal"}

    # The training_attack_types file stores one attack name and one family per line.
    for raw_line in attack_types_path.read_text().splitlines():
        cleaned_line = raw_line.strip()
        if not cleaned_line:
            continue
        attack_name, attack_family = cleaned_line.split()
        attack_family_map[f"{attack_name}."] = family_name_map[attack_family.lower()]

    return attack_family_map


attack_family_map = load_attack_family_map(ATTACK_TYPES_PATH)
df["attack_family"] = df["label"].map(attack_family_map).fillna("unknown")

# Use a stable ordering so repeated runs are easier to compare in the report.
family_order = ["normal", "DoS", "Probe", "R2L", "U2R", "unknown"]
attack_family_counts = df["attack_family"].value_counts().reindex(family_order, fill_value=0)
attack_family_share = attack_family_counts.div(len(df)).mul(100).round(2)

print("Attack-family distribution:")
print(pd.DataFrame({"count": attack_family_counts, "share_pct": attack_family_share}))

plt.figure(figsize=(10, 5))
sns.barplot(x=attack_family_counts.index, y=attack_family_counts.values, palette="mako")
plt.title("KDD99 Traffic by Attack Family")
plt.xlabel("Attack Family")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()


## 5. Class Imbalance Analysis

KDD99 is highly imbalanced. The first plot compresses the labels into normal versus attack traffic. The second plot shows the most common attack labels.


In [ ]:
# Collapse the labels into normal versus attack to show the high-level imbalance first.
traffic_class = np.where(df["label"] == "normal.", "normal", "attack")
traffic_class_counts = pd.Series(traffic_class).value_counts()
label_counts = df["label"].value_counts()
attack_counts = df.loc[df["label"] != "normal.", "label"].value_counts()

print("Binary traffic-class distribution:")
print(traffic_class_counts)
print()
print("Label distribution (top 15):")
print(label_counts.head(15))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(x=traffic_class_counts.index, y=traffic_class_counts.values, ax=axes[0], palette="Set2")
axes[0].set_title("Normal vs Attack Traffic")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Number of Records")

# A log x-axis keeps the long-tailed attack labels readable.
top_attack_counts = attack_counts.head(15)
sns.barplot(x=top_attack_counts.values, y=top_attack_counts.index, ax=axes[1], palette="flare")
axes[1].set_title("Top Attack Types")
axes[1].set_xlabel("Number of Records")
axes[1].set_ylabel("Attack Type")
axes[1].set_xscale("log")

plt.tight_layout()
plt.show()


## 6. Missing Values and Duplicate Records

These checks help confirm whether the dataset needs cleaning before training the autoencoder.


In [ ]:
# Missing values are uncommon in KDD99, but this confirms whether imputation is needed.
missing_values = df.isna().sum().sort_values(ascending=False)
duplicate_count = df.duplicated().sum()

print("Missing values per column:")
missing_value_summary = missing_values[missing_values > 0]
if missing_value_summary.empty:
    print("No missing values found.")
else:
    print(missing_value_summary)

print()
print(f"Duplicate rows: {duplicate_count}")
if duplicate_count > 0:
    duplicate_share = duplicate_count / len(df)
    print(f"Duplicate share: {duplicate_share:.2%}")


## 7. Zero-Variance Feature Detection

Constant columns add no predictive signal and are strong candidates for removal before scaling and autoencoder training.


In [ ]:
# Columns with only one unique value can be dropped before modeling.
numeric_df = df.select_dtypes(include=["number"])
zero_variance_features = [column for column in numeric_df.columns if numeric_df[column].nunique(dropna=False) == 1]

if zero_variance_features:
    zero_variance_summary = pd.DataFrame(
        {"constant_value": [numeric_df[column].iloc[0] for column in zero_variance_features]},
        index=zero_variance_features,
    )
    print("Zero-variance numeric features:")
    print(zero_variance_summary)
else:
    zero_variance_summary = pd.DataFrame(columns=["constant_value"])
    print("No zero-variance numeric features found.")


## 8. Categorical Cardinality

Rare categories can make one-hot encoding unnecessarily wide, especially in the `service` feature. This section highlights the cardinality and the low-frequency service values that may need grouping.


In [ ]:
# Use the data dictionary to avoid hardcoding categorical columns in the notebook.
categorical_cardinality = df[categorical_features].nunique().sort_values(ascending=False)
print("Categorical feature cardinality:")
print(categorical_cardinality)

# Treat values that appear in fewer than 0.1% of rows as rare categories.
rare_service_threshold = max(10, int(len(df) * 0.001))
service_counts = df["service"].value_counts()
rare_services = service_counts[service_counts < rare_service_threshold]

print()
print(f"Rare service threshold: < {rare_service_threshold} rows")
print(f"Number of rare services: {len(rare_services)}")
if not rare_services.empty:
    print(rare_services.head(20))

plt.figure(figsize=(12, 8))
sns.barplot(y=service_counts.head(20).index, x=service_counts.head(20).values, palette="crest")
plt.title("Top 20 Services by Frequency")
plt.xlabel("Number of Records")
plt.ylabel("Service")
plt.tight_layout()
plt.show()


## 9. Skewness and Outlier Analysis

Network traffic features are often long-tailed. This section identifies the most skewed continuous features, estimates the fraction of IQR-defined outliers, and visualizes raw versus log-transformed distributions.


In [ ]:
# Remove constant features first so they do not dominate the skewness summary.
continuous_feature_df = numeric_df.drop(columns=zero_variance_features, errors="ignore")
skewness_summary = continuous_feature_df.skew().sort_values(key=lambda series: series.abs(), ascending=False)
top_skewed_features = skewness_summary.head(6)

def iqr_outlier_share(series: pd.Series) -> float:
    """Estimate the outlier share using the 1.5 * IQR rule."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return ((series < lower_bound) | (series > upper_bound)).mean()

outlier_summary = pd.DataFrame({
    "skewness": top_skewed_features,
    "iqr_outlier_share": [iqr_outlier_share(continuous_feature_df[column]) for column in top_skewed_features.index],
}).sort_values("skewness", key=lambda series: series.abs(), ascending=False)

print("Most skewed numeric features:")
print(outlier_summary)

# Compare the raw and log1p-transformed shapes for the most skewed strictly nonnegative features.
plot_features = [column for column in top_skewed_features.index if continuous_feature_df[column].min() >= 0][:4]
fig, axes = plt.subplots(len(plot_features), 2, figsize=(14, 4 * len(plot_features)))
if len(plot_features) == 1:
    axes = np.array([axes])

for row_index, feature_name in enumerate(plot_features):
    sns.histplot(continuous_feature_df[feature_name], bins=50, ax=axes[row_index, 0], color="steelblue")
    axes[row_index, 0].set_title(f"{feature_name} distribution")
    axes[row_index, 0].set_xlabel(feature_name)

    sns.histplot(np.log1p(continuous_feature_df[feature_name]), bins=50, ax=axes[row_index, 1], color="darkorange")
    axes[row_index, 1].set_title(f"log1p({feature_name}) distribution")
    axes[row_index, 1].set_xlabel(f"log1p({feature_name})")

plt.tight_layout()
plt.show()


## 10. Correlation Matrix for Continuous Features

The numeric columns are plotted below so we can identify strong positive or negative relationships before feature scaling and model training. Constant features are excluded to keep the heatmap readable.


In [ ]:
# Drop constant columns before computing correlations to avoid all-NaN rows and columns.
correlation_input_df = numeric_df.drop(columns=zero_variance_features, errors="ignore")
correlation_matrix = correlation_input_df.corr()

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
plt.figure(figsize=(18, 14))
sns.heatmap(
    correlation_matrix,
    mask=mask,
    cmap="coolwarm",
    center=0,
    linewidths=0.2,
    cbar_kws={"shrink": 0.75},
)
plt.title("Correlation Matrix for Numeric KDD99 Features")
plt.tight_layout()
plt.show()
